# Company Rule Book Chatbot: a basic RAG pipeline

This notebook builds a working question-answering chatbot over a company rule book (PDF), one step at a time:

**Load → Read → Clean → Chunk → Embed → Store (vector DB) → Retrieve → Generate**

| Step | Tool used | Topic |
|---|---|---|
| Load & read the PDF | `pypdf` | 3.1 |
| Clean the text | Python `re` | 3.1 |
| Chunk | `RecursiveCharacterTextSplitter` | 3.1 |
| Embed | `sentence-transformers` (all-MiniLM-L6-v2) | 3.2 |
| Store | `ChromaDB` | 3.2 |
| Retrieve | top-k similarity search | 3.3 |
| Generate | an LLM (Claude), with a no-API-key fallback | RAG |

## Step 0: Install

In [1]:
# Run once, then restart the kernel if asked
# !pip install -q pypdf langchain-text-splitters sentence-transformers chromadb anthropic reportlab

In [2]:
import os, re, textwrap, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
PDF_PATH = "company_rulebook.pdf"     # change this to your own PDF

## Step 1 (optional): Create a sample rule book PDF
Skip this cell if you already have a PDF. It creates a 3-page rule book with the same mess real PDFs have: a header and footer on every page, words split with hyphens across lines, and extra spaces.

In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

pages = [
    ["1. LEAVE POLICY",
     "1.1 All full-time employees are entitled to 24 days of paid annual",
     "leave per calendar year. Leave must be applied for on the HR portal at",
     "least 7 days in advance.",
     "1.2 Up to 10 unused annual leave days can be carried forward to the next",
     "year. Any remaining days lapse on 31 March.",
     "1.3 Sick leave is 8 days per year. A medical certificate is required for",
     "absences longer than 2 consecu-",
     "tive days.",
     "1.4 Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks.",
     "",
     "2. WORKING HOURS AND ATTENDANCE",
     "2.1 Office hours are 9:30 am to 6:00 pm, Monday to Friday.",
     "2.2 Employees may work from home up to two days per week with",
     "manager approval.",
     "2.3 Arriving more than 15 minutes late three times in a month is",
     "treated as a half-day of leave."],
    ["3. TRAVEL AND EXPENSES",
     "3.1 Economy class is used for all flights shorter than 6 hours. Business",
     "class is allowed only for international flights longer than 6 hours,",
     "with director approval.",
     "3.2 Hotel bookings must be made through the company travel desk at",
     "least 3 days in advance.",
     "3.3 The daily meal allowance during business travel is Rs 1,500 in metro",
     "cities and Rs 1,000 elsewhere.",
     "3.4 Expense claims must be submitted within 30 days with original re-",
     "ceipts. Approved claims are paid with the next monthly salary.",
     "",
     "4. IT AND DATA SECURITY",
     "4.1 Passwords must be changed every 90 days and contain at least 12",
     "characters.",
     "4.2 Laptops must be locked whenever they are left unattended.",
     "4.3 Company data must never be uploaded to personal cloud storage or",
     "shared on personal email."],
    ["5. CODE OF CONDUCT",
     "5.1 Gifts from vendors worth more than Rs 2,000 must be declared to the",
     "compliance team within 7 days.",
     "5.2 Complaints of harassment are handled confidentially by the Internal",
     "Committee. Email ic@novaretail.in.",
     "5.3 The dress code is business casual from Monday to Thursday and",
     "casual on Fridays.",
     "",
     "6. BENEFITS",
     "6.1 Health insurance covers the employee, spouse, two children and",
     "parents up to Rs 5 lakh per year.",
     "6.2 Employees get a 20% discount on all Nova Retail products.",
     "6.3 Performance reviews take place twice a year, in June and December.",
     "Employees rated 'exceeds expectations' may receive a bonus of up to",
     "15% of annual salary."],
]

c = canvas.Canvas(PDF_PATH, pagesize=A4)
W, H = A4
for n, lines in enumerate(pages, 1):
    c.setFont("Helvetica", 8)
    c.drawString(50, H - 40, "NOVA RETAIL PVT LTD  |  EMPLOYEE RULE BOOK 2025  |  CONFIDENTIAL")   # header
    c.drawString(50, 30, f"Page {n} of {len(pages)}")                                              # footer
    c.setFont("Helvetica", 11)
    y = H - 90
    for line in lines:
        c.drawString(60, y, line.replace(". ", ".   ") if n == 2 and line.startswith("3.2") else line)
        y -= 18
    c.showPage()
c.save()
print("Created", PDF_PATH)

Created company_rulebook.pdf


## Step 2: Load the PDF
`pypdf` opens the file and gives us one page object per page. We keep the **page number** with each page's text so the chatbot can later say *where* an answer came from.

In [4]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
print("Number of pages:", len(reader.pages))

raw_pages = []
for i, page in enumerate(reader.pages, start=1):
    raw_pages.append({"page": i, "text": page.extract_text() or ""})

Number of pages: 3


## Step 3: Read (inspect) the raw text
Always look at the raw text before cleaning. Notice the header, the footer, the broken word "consecu- tive" and extra spaces.

In [5]:
print(raw_pages[0]["text"][:900])
print("\n... total characters across all pages:", sum(len(p["text"]) for p in raw_pages))

NOVA RETAIL PVT LTD  |  EMPLOYEE RULE BOOK 2025  |  CONFIDENTIAL
Page 1 of 3
1. LEAVE POLICY
1.1 All full-time employees are entitled to 24 days of paid annual
leave per calendar year. Leave must be applied for on the HR portal at
least 7 days in advance.
1.2 Up to 10 unused annual leave days can be carried forward to the next
year. Any remaining days lapse on 31 March.
1.3 Sick leave is 8 days per year. A medical certificate is required for
absences longer than 2 consecu-
tive days.
1.4 Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks.
2. WORKING HOURS AND ATTENDANCE
2.1 Office hours are 9:30 am to 6:00 pm, Monday to Friday.
2.2 Employees may work from home up to two days per week with
manager approval.
2.3 Arriving more than 15 minutes late three times in a month is
treated as a half-day of leave.


... total characters across all pages: 2415


## Step 4: Clean the text
Cleaning rules (adjust them to your own PDF):
1. Remove the repeated **header** and **footer** lines
2. Join words **hyphenated across lines** (`consecu-\ntive` → `consecutive`)
3. Join lines that were broken in the middle of a sentence
4. Collapse multiple spaces

In [6]:
def clean_text(text):
    lines = text.split("\n")
    # 1. drop header / footer lines
    lines = [l for l in lines
             if not re.search(r"EMPLOYEE RULE BOOK|CONFIDENTIAL", l)
             and not re.match(r"\s*Page \d+ of \d+\s*$", l)]
    text = "\n".join(lines)
    # 2. join hyphenated words split across lines
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # 3. a line break NOT followed by a new rule number or heading = same sentence
    text = re.sub(r"\n(?!\d+\.\d*\s|\d+\.\s|\n)", " ", text)
    # 4. tidy spaces
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

pages = [{"page": p["page"], "text": clean_text(p["text"])} for p in raw_pages]

print("BEFORE:\n", raw_pages[0]["text"][:400])
print("\nAFTER:\n", pages[0]["text"][:400])

BEFORE:
 NOVA RETAIL PVT LTD  |  EMPLOYEE RULE BOOK 2025  |  CONFIDENTIAL
Page 1 of 3
1. LEAVE POLICY
1.1 All full-time employees are entitled to 24 days of paid annual
leave per calendar year. Leave must be applied for on the HR portal at
least 7 days in advance.
1.2 Up to 10 unused annual leave days can be carried forward to the next
year. Any remaining days lapse on 31 March.
1.3 Sick leave is 8 days pe

AFTER:
 1. LEAVE POLICY
1.1 All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave must be applied for on the HR portal at least 7 days in advance.
1.2 Up to 10 unused annual leave days can be carried forward to the next year. Any remaining days lapse on 31 March.
1.3 Sick leave is 8 days per year. A medical certificate is required for absences longer than 2 consecut


## Step 5: Chunk the text
We split each page into chunks of about **250 characters** with **40 characters of overlap**, so a rule that falls on a chunk boundary is not cut in half. `RecursiveCharacterTextSplitter` tries to split on paragraphs first, then lines, then sentences, then words.

Each chunk keeps its **page number** as metadata.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for p in pages:
    for j, piece in enumerate(splitter.split_text(p["text"])):
        chunks.append({"id": f"p{p['page']}_c{j}", "text": piece, "page": p["page"]})

print("Total chunks:", len(chunks))
for ch in chunks[:3]:
    print(f"\n[{ch['id']}] (page {ch['page']}, {len(ch['text'])} chars)\n{ch['text']}")

Total chunks: 11

[p1_c0] (page 1, 178 chars)
1. LEAVE POLICY
1.1 All full-time employees are entitled to 24 days of paid annual leave per calendar year. Leave must be applied for on the HR portal at least 7 days in advance.

[p1_c1] (page 1, 230 chars)
1.2 Up to 10 unused annual leave days can be carried forward to the next year. Any remaining days lapse on 31 March.
1.3 Sick leave is 8 days per year. A medical certificate is required for absences longer than 2 consecutive days.

[p1_c2] (page 1, 245 chars)
1.4 Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks.
2. WORKING HOURS AND ATTENDANCE
2.1 Office hours are 9:30 am to 6:00 pm, Monday to Friday.
2.2 Employees may work from home up to two days per week with manager approval.


## Step 6: Embed the chunks
Each chunk becomes a vector of **384 numbers** that captures its meaning. Similar meanings → similar vectors.

In [8]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [ch["text"] for ch in chunks]
vectors = embedder.encode(texts, normalize_embeddings=True)

print("Embedding matrix shape:", vectors.shape)    # (number of chunks, 384)
print("First 5 numbers of chunk 0:", vectors[0][:5].round(3))

Embedding matrix shape: (11, 384)
First 5 numbers of chunk 0: [0.058 0.051 0.047 0.02  0.085]


## Step 7: Store in a vector database (Chroma)
We save the **id, vector, text and page number** together. `PersistentClient` writes the database to a folder, so next time you can skip steps 2–7 and just load it.

In [9]:
import chromadb

db = chromadb.PersistentClient(path="./rulebook_db")
try:
    db.delete_collection("rulebook")          # start fresh each time this cell runs
except Exception:
    pass
collection = db.create_collection("rulebook", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[ch["id"] for ch in chunks],
    embeddings=vectors.tolist(),
    documents=texts,
    metadatas=[{"page": ch["page"]} for ch in chunks],
)
print("Chunks stored:", collection.count())

Chunks stored: 11


## Step 8: Retrieve the most relevant chunks
Embed the question with the **same model**, then ask Chroma for the closest chunks. Chroma returns a *distance*; with cosine space, `similarity = 1 − distance`.

In [10]:
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q_vec, n_results=k)
    return [{"text": doc, "page": meta["page"], "score": round(1 - dist, 3)}
            for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

for hit in retrieve("How many days of leave do I get?"):
    print(f"score={hit['score']}  page={hit['page']}  {hit['text'][:110]}...")

score=0.749  page=1  1.2 Up to 10 unused annual leave days can be carried forward to the next year. Any remaining days lapse on 31 ...
score=0.677  page=1  1. LEAVE POLICY
1.1 All full-time employees are entitled to 24 days of paid annual leave per calendar year. Le...
score=0.547  page=1  2.3 Arriving more than 15 minutes late three times in a month is treated as a half-day of leave....


## Step 9: Generate the answer (the "G" in RAG)
We put the retrieved chunks into a prompt and tell the LLM to answer **only** from them and to cite the page.

**Using an API key:** set `ANTHROPIC_API_KEY` (e.g. in Colab: `os.environ["ANTHROPIC_API_KEY"] = "..."`). To use another provider, only `call_llm()` changes.

**No API key?** The notebook still runs: it shows the prompt and the best matching rule as a simple answer.

In [11]:
def build_prompt(question, hits):
    context = "\n\n".join(f"[Page {h['page']}] {h['text']}" for h in hits)
    return f"""You are the HR assistant for Nova Retail. Answer the employee's question using ONLY
the rule book extracts below. Keep the answer short and cite the page like (Page 2).
If the answer is not in the extracts, say: "I couldn't find this in the rule book. Please contact HR."

Rule book extracts:
{context}

Question: {question}
Answer:"""

def call_llm(prompt):
    import anthropic
    client = anthropic.Anthropic()            # reads ANTHROPIC_API_KEY
    msg = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print(build_prompt("Can I fly business class?", retrieve("Can I fly business class?")))

You are the HR assistant for Nova Retail. Answer the employee's question using ONLY
the rule book extracts below. Keep the answer short and cite the page like (Page 2).
If the answer is not in the extracts, say: "I couldn't find this in the rule book. Please contact HR."

Rule book extracts:
[Page 2] 3. TRAVEL AND EXPENSES
3.1 Economy class is used for all flights shorter than 6 hours. Business class is allowed only for international flights longer than 6 hours, with director approval.

[Page 2] 3.2 Hotel bookings must be made through the company travel desk at least 3 days in advance.
3.3 The daily meal allowance during business travel is Rs 1,500 in metro cities and Rs 1,000 elsewhere.

[Page 3] 5. CODE OF CONDUCT
5.1 Gifts from vendors worth more than Rs 2,000 must be declared to the compliance team within 7 days.
5.2 Complaints of harassment are handled confidentially by the Internal Committee. Email ic@novaretail.in.

Question: Can I fly business class?
Answer:


## Step 10: The chatbot
`ask()` joins everything: retrieve → check relevance → build prompt → generate. A **similarity threshold** stops the bot from answering questions the rule book doesn't cover.

In [12]:
THRESHOLD = 0.30     # below this, the best chunk is probably not relevant

def ask(question, k=3, show_sources=True):
    hits = retrieve(question, k)
    if hits[0]["score"] < THRESHOLD:
        answer = "I couldn't find this in the rule book. Please contact HR."
    elif os.getenv("ANTHROPIC_API_KEY"):
        answer = call_llm(build_prompt(question, hits))
    else:
        # fallback without an LLM: return the best matching rule
        answer = f"(No API key, showing the best matching rule) {hits[0]['text']} (Page {hits[0]['page']})"
    print("Q:", question)
    print("A:", textwrap.fill(answer, 100))
    if show_sources:
        print("   sources:", [(h["page"], h["score"]) for h in hits])
    print()
    return answer

ask("How many sick days do I get?")
ask("Can I fly business class to London?")
ask("What is the meal allowance in Mumbai?")
ask("Can I bring my dog to the office?")

Q: How many sick days do I get?
A: (No API key, showing the best matching rule) 1.2 Up to 10 unused annual leave days can be carried
forward to the next year. Any remaining days lapse on 31 March. 1.3 Sick leave is 8 days per year. A
medical certificate is required for absences longer than 2 consecutive days. (Page 1)
   sources: [(1, 0.615), (1, 0.319), (1, 0.311)]

Q: Can I fly business class to London?
A: (No API key, showing the best matching rule) 3. TRAVEL AND EXPENSES 3.1 Economy class is used for
all flights shorter than 6 hours. Business class is allowed only for international flights longer
than 6 hours, with director approval. (Page 2)
   sources: [(2, 0.564), (2, 0.34), (3, 0.165)]

Q: What is the meal allowance in Mumbai?
A: (No API key, showing the best matching rule) 3.2 Hotel bookings must be made through the company
travel desk at least 3 days in advance. 3.3 The daily meal allowance during business travel is Rs
1,500 in metro cities and Rs 1,000 elsewhere. (Page 2)
  

"I couldn't find this in the rule book. Please contact HR."

### Chat loop
Uncomment to chat in the notebook. Type `exit` to stop.

In [13]:
# while True:
#     q = input("You: ")
#     if q.strip().lower() in {"exit", "quit"}:
#         break
#     ask(q, show_sources=False)

## Re-using the database later
Once Step 7 has run, the vectors are saved in `./rulebook_db`. In a new session you only need:
```python
import chromadb
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
collection = chromadb.PersistentClient(path="./rulebook_db").get_collection("rulebook")
# then define retrieve(), build_prompt(), call_llm(), ask() and start asking
```

## Try it yourself
1. Replace `PDF_PATH` with a real policy PDF and adjust `clean_text()` for its header and footer.
2. Change `chunk_size` to 100 and 1000. How do the answers change?
3. Ask 5 questions and check whether the correct page appears in `sources`.
4. Add the section heading (e.g. "3. TRAVEL AND EXPENSES") to each chunk's metadata.
5. Wrap `ask()` in a simple web app with `gradio`: `gr.Interface(fn=ask, inputs="text", outputs="text").launch()`.